# Park Arms Action Designator

Demonstrates `ParkArmsActionDescription` — returning arms to the safe resting position.

Available options: `Arms.LEFT`, `Arms.RIGHT`, `Arms.BOTH`

**Prerequisites:** `cd cognitive_robot_abstract_machine && uv sync --active`

## 1 · Imports

In [1]:
import logging
logging.basicConfig(level=logging.WARNING)

from semantic_digital_twin.world import World
from semantic_digital_twin.robots.pr2 import PR2

from pycram.datastructures.dataclasses import Context
from pycram.datastructures.enums import Arms
from pycram.language import SequentialPlan
from pycram.motion_executor import simulated_robot
from pycram.testing import setup_world
from pycram.robot_plans import ParkArmsActionDescription

print('All imports OK')

/home/malineni/workingdir/cognitive_robot_abstract_machine/semantic_digital_twin/src/semantic_digital_twin/robots/pr2.py:8: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_filename


Found these qp solvers: ['qpSWIFT', 'qpalm']


All imports OK


## 2 · World & Context Setup

In [2]:
world = setup_world()

try:
    robot = PR2.from_world(world)
except Exception as exc:
    print(f'Note: PR2 full setup skipped ({type(exc).__name__}) — recovering annotation')
    robot = next((a for a in world.semantic_annotations if isinstance(a, PR2)), None)
    if robot is None:
        raise RuntimeError('Could not obtain PR2 annotation') from exc
    with world.modify_world():
        robot._setup_velocity_limits()
        robot._setup_hardware_interfaces()
        robot._setup_joint_states()

context = Context(world, robot)
print('World ready')

Unknown attribute "type" in /robot[@name='pr2']/link[@name='base_laser_link']
Unknown attribute "type" in /robot[@name='pr2']/link[@name='wide_stereo_optical_frame']
Unknown attribute "type" in /robot[@name='pr2']/link[@name='narrow_stereo_optical_frame']
Unknown attribute "type" in /robot[@name='pr2']/link[@name='laser_tilt_link']
INFO:semantic_digital_twin.world:Trying to add kinematic_structure_entity with name pr2/base_footprint
INFO:semantic_digital_twin.world:Trying to add kinematic_structure_entity with name pr2/base_footprint
INFO:semantic_digital_twin.world:Trying to add kinematic_structure_entity with name pr2/base_link
INFO:semantic_digital_twin.world:Trying to add kinematic_structure_entity with name pr2/base_link
INFO:semantic_digital_twin.world:Trying to add kinematic_structure_entity with name pr2/base_bellow_link
INFO:semantic_digital_twin.world:Trying to add kinematic_structure_entity with name pr2/base_link
INFO:semantic_digital_twin.world:Trying to add kinematic_stru

Note: PR2 full setup skipped (WorldEntityNotFoundError) — recovering annotation
World ready


## 3 · Visualize in RViz2 (optional)

Skip this section if ROS2 is not available.

In [3]:
from semantic_digital_twin.adapters.ros.tf_publisher import TFPublisher
from semantic_digital_twin.adapters.ros.visualization.viz_marker import VizMarkerPublisher
import threading
import rclpy

rclpy.init()
_ros_node = rclpy.create_node('semantic_digital_twin')
_ros_thread = threading.Thread(target=rclpy.spin, args=(_ros_node,), daemon=True)
_ros_thread.start()

In [4]:
_tf_publisher  = TFPublisher(_world=world, node=_ros_node)
_viz_publisher = VizMarkerPublisher(_world=world, node=_ros_node)
print('ROS2 publishers started')
print('  Fixed Frame : apartment/apartment_root')
print('  TF topic    : /tf')
print('  Marker topic: /semworld/viz_marker  (QoS Durability=Transient Local)')

ROS2 publishers started
  Fixed Frame : apartment/apartment_root
  TF topic    : /tf
  Marker topic: /semworld/viz_marker  (QoS Durability=Transient Local)


---
## 4 · Park Both Arms

In [5]:
with simulated_robot:
    SequentialPlan(
        context,
        ParkArmsActionDescription(Arms.BOTH),
    ).perform()

print('Park Arms BOTH done')

INFO:pycram.language:Executing SequentialNode
INFO:pycram.robot_plans.actions.base:Performing action ParkArmsAction
INFO:pycram.language:Executing SequentialNode


Park Arms BOTH done


## 5 · Park Right Arm Only

In [6]:
with simulated_robot:
    SequentialPlan(
        context,
        ParkArmsActionDescription(Arms.RIGHT),
    ).perform()

print('Park Arms RIGHT done')

INFO:pycram.language:Executing SequentialNode
INFO:pycram.robot_plans.actions.base:Performing action ParkArmsAction
INFO:pycram.language:Executing SequentialNode


Park Arms RIGHT done


## 6 · Park Left Arm Only

In [7]:
with simulated_robot:
    SequentialPlan(
        context,
        ParkArmsActionDescription(Arms.LEFT),
    ).perform()

print('Park Arms LEFT done')

INFO:pycram.language:Executing SequentialNode
INFO:pycram.robot_plans.actions.base:Performing action ParkArmsAction
INFO:pycram.language:Executing SequentialNode


Park Arms LEFT done


---
## Shutdown ROS2 Node

In [8]:
try:
    _ros_node.destroy_node()
    rclpy.shutdown()
    print('ROS2 node stopped')
except Exception:
    print('(ROS2 node was not running)')

ROS2 node stopped
